<a href="https://colab.research.google.com/github/aaditjoshi-star/Aadit/blob/main/Exercise2_ReAct_Code_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2: Code Generation with ReAct Prompting
**Aadit Joshi** | Agentic AI and Distributed Systems | San José State University | Fall 2026

**Goal:** Use a ReAct-style prompt (reasoning + acting) to have an AI model write Python code, run that code, observe the result, and fix it until it works.

**Tools used**
- Google Colab (Python 3)
- Gemini API through the `google-genai` Python SDK, free tier (first choice `gemini-3.8-flash`, with other free Flash models as fallbacks if a daily limit is hit)
- Python `subprocess` to run each piece of generated code in a separate process (the "act" part of ReAct), and an automatic test suite to check the final function

**How to run**
1. Create a free API key at https://aistudio.google.com/apikey
2. In Colab, open **Secrets** (key icon in the left sidebar), add `GEMINI_API_KEY`, and turn on notebook access.
3. Runtime > Run all. (The ReAct loop makes several API calls; if the free tier rate-limits you, the notebook waits and retries automatically.)

## How the ReAct loop works

```
 PLAN  -->  THOUGHT  -->  ACTION (Python code)  -->  code runs in Colab  -->  OBSERVATION (output or error)
               ^                                                                   |
               |                                                                   v
               +---------------  FIX (name the cause, change the code)  <----------+

 FINAL_ANSWER (complete function)  -->  automatic tests  -->  all pass: done
                                                          -->  any fail: failures come back as an OBSERVATION, loop continues
```

The model **never sees the data file up front.** It has to act (write code that inspects the file), observe what comes back, and reason about it before writing the final function. That is what makes this ReAct rather than one-shot code generation.

## Setup

In [4]:
!pip install -q google-genai

In [5]:
# Setup: Gemini client and shared helper functions
import os, json, re, sys, time, shutil, subprocess, textwrap
from pathlib import Path
from google import genai
from google.genai import types, errors

# API key is stored in Colab Secrets (key icon in the left sidebar) as GEMINI_API_KEY
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass  # not running in Colab: set the GEMINI_API_KEY environment variable instead

client = genai.Client()  # reads GEMINI_API_KEY

# First-choice model plus free-tier fallbacks. Each model has its own daily quota,
# so if one runs out the notebook moves to the next one.
MODELS = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash",
          "gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]
model_index = 0
MODELS_USED = set()
RETRYABLE = {429, 500, 503}


def generate(contents, system=None, max_retries=4):
    """Send a prompt (or a multi-turn history) to Gemini and return the reply text.
    Retries on rate limits / server errors and falls back to the next model if needed."""
    global model_index
    config = types.GenerateContentConfig(system_instruction=system)
    attempt = 0
    while True:
        model = MODELS[model_index]
        try:
            response = client.models.generate_content(model=model, contents=contents, config=config)
            if response.text and response.text.strip():
                MODELS_USED.add(model)
                return response.text.strip()
            code, detail = "empty", "empty response"
        except errors.APIError as e:
            code, detail = e.code, str(e)
            if code not in RETRYABLE and code != 404:
                raise
        attempt += 1
        out_of_daily_quota = code == 429 and "perday" in detail.lower().replace(" ", "")
        if code == 404 or out_of_daily_quota or attempt > max_retries:
            if model_index + 1 >= len(MODELS):
                raise RuntimeError(f"All models failed. Last error: {detail[:300]}")
            model_index += 1
            attempt = 0
            print(f"   [{model} unavailable ({code}); switching to {MODELS[model_index]}]")
        else:
            wait = 15 * attempt
            print(f"   [{model} returned {code}; retrying in {wait}s]")
            time.sleep(wait)


def user_turn(text):
    return {"role": "user", "parts": [{"text": text}]}


def model_turn(text):
    return {"role": "model", "parts": [{"text": text}]}


def wrap(text, width=100):
    """Wrap long prose lines so outputs are readable on GitHub (code lines are left alone)."""
    out = []
    for line in str(text).splitlines():
        out.append(line if len(line) <= width or line.startswith((" ", "\t")) else textwrap.fill(line, width))
    return "\n".join(out)


def show(title, body=""):
    print("\n" + "=" * 100)
    print(title)
    print("-" * 100)
    if body != "":
        print(wrap(body) if isinstance(body, str) else json.dumps(body, indent=2))

# Quick connectivity check
print(generate("Reply with exactly one word: ready"))
print("Model in use:", MODELS[model_index])

## Test data: a messy sales export

`sales.csv` is fictional order data for "NimbusCart" (the same made-up store as Exercise 1). It has the kinds of problems real exports have: three date formats, prices written as `$1,299.00` or with extra spaces, blank or non-numeric quantities, a duplicated order, inconsistent category capitalization, and cancelled/refunded orders.

The file is printed below **for the reader only**. The model is never shown it; it has to discover these problems through ACTION steps.

In [6]:
WORKDIR = Path("react_workspace")   # the only folder the model's code runs in
TESTDIR = Path("hidden_tests")      # test files the model never sees
for folder in (WORKDIR, TESTDIR):
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir()

SALES_CSV = """order_id,order_date,product,category,quantity,unit_price,status
1001,2026-07-03,Laptop Stand,Accessories,2,24.99,completed
1002,07/15/2026,UltraBook 14,Electronics,1,"$1,299.00",completed
1003,Jul 21 2026,USB-C Hub,accessories ,3,19.50,Completed
1004,2026-07-28,Desk Lamp,Home Office,1,39.99,cancelled
1005,08/02/2026,Noise-Cancelling Headphones,ELECTRONICS,1,$249.00,completed
1005,08/02/2026,Noise-Cancelling Headphones,ELECTRONICS,1,$249.00,completed
1006,Aug 9 2026,Ergonomic Chair,home office,1,"$1,150.00",completed
1007,2026-08-14,Laptop Stand,Accessories,1,N/A,completed
1008,08/19/2026,USB-C Hub,Accessories,,19.50,completed
1009,Aug 23 2026,Webcam HD,Electronics,2, 59.90 ,completed
1010,2026-08-30,Desk Lamp, Home Office,2,39.99,COMPLETED
1011,09/03/2026,UltraBook 14,Electronics,1,"$1,299.00",Cancelled
1012,Sep 7 2026,Laptop Stand,accessories,4,24.99,completed
1013,2026-09-12,Webcam HD,Electronics,two,59.90,completed
1014,09/18/2026,Noise-Cancelling Headphones,Electronics,1,$249.00,completed
1015,Sep 21 2026,USB-C Hub ,Accessories,5,19.50,completed
1016,2026-09-24,Ergonomic Chair,Home Office,1,"$1,150.00",refunded
1017,09/25/2026,Desk Lamp,home office,1,$39.99,completed
1018,Sep 26 2026,Laptop Stand,Accessories,0,24.99,completed
"""

# A second file with the same kinds of problems but different numbers.
# If the model hard-codes answers for sales.csv, the test on this file fails.
CHECK_CSV = """order_id,order_date,product,category,quantity,unit_price,status
2001,2026-10-01,Webcam HD,Electronics,3,$59.90,completed
2002,10/05/2026,Desk Lamp, home office ,2,39.99,Completed
2002,10/05/2026,Desk Lamp, home office ,2,39.99,Completed
2003,Oct 7 2026,USB-C Hub,ACCESSORIES,,19.50,completed
2004,11/02/2026,UltraBook 14,Electronics,1,"$1,299.00",cancelled
2005,Nov 3 2026,Laptop Stand,Accessories,3, 24.99 ,completed
2006,2026-11-09,Ergonomic Chair,Home Office,1,N/A,completed
"""
EMPTY_CSV = "order_id,order_date,product,category,quantity,unit_price,status\n"

(WORKDIR / "sales.csv").write_text(SALES_CSV)
(TESTDIR / "sales.csv").write_text(SALES_CSV)
(TESTDIR / "check.csv").write_text(CHECK_CSV)
(TESTDIR / "empty.csv").write_text(EMPTY_CSV)
print(SALES_CSV)

order_id,order_date,product,category,quantity,unit_price,status
1001,2026-07-03,Laptop Stand,Accessories,2,24.99,completed
1002,07/15/2026,UltraBook 14,Electronics,1,"$1,299.00",completed
1003,Jul 21 2026,USB-C Hub,accessories ,3,19.50,Completed
1004,2026-07-28,Desk Lamp,Home Office,1,39.99,cancelled
1005,08/02/2026,Noise-Cancelling Headphones,ELECTRONICS,1,$249.00,completed
1005,08/02/2026,Noise-Cancelling Headphones,ELECTRONICS,1,$249.00,completed
1006,Aug 9 2026,Ergonomic Chair,home office,1,"$1,150.00",completed
1007,2026-08-14,Laptop Stand,Accessories,1,N/A,completed
1008,08/19/2026,USB-C Hub,Accessories,,19.50,completed
1009,Aug 23 2026,Webcam HD,Electronics,2, 59.90 ,completed
1010,2026-08-30,Desk Lamp, Home Office,2,39.99,COMPLETED
1011,09/03/2026,UltraBook 14,Electronics,1,"$1,299.00",Cancelled
1012,Sep 7 2026,Laptop Stand,accessories,4,24.99,completed
1013,2026-09-12,Webcam HD,Electronics,two,59.90,completed
1014,09/18/2026,Noise-Cancelling Headphones,Electronics,1,$249.00,co

## Reference answer and automatic tests (hidden from the model)

To judge the model's code objectively, I wrote my own reference solution and computed the correct answers from it. The model's final function must match them. The tests check:

1. the code imports without errors
2. the correct summary of `sales.csv`
3. the correct summary of a **different** file (catches hard-coded answers)
4. a header-only file returns an empty summary (edge case)
5. a missing file raises `FileNotFoundError` that names the path (error handling)

In [7]:
%%writefile hidden_tests/reference_solution.py
# My own solution. It is used only to produce the expected answers for the tests; the model never sees it.
import csv
import os
from collections import defaultdict
from datetime import datetime

DATE_FORMATS = ["%Y-%m-%d", "%m/%d/%Y", "%b %d %Y"]


def _positive_number(text):
    # Convert '$1,299.00', ' 59.90 ', '2' to a float; return None if impossible or not > 0
    try:
        value = float(str(text).replace("$", "").replace(",", "").strip())
    except ValueError:
        return None
    return value if value > 0 else None


def _parse_date(text):
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(text.strip(), fmt)
        except ValueError:
            pass
    raise ValueError(f"Unrecognized date: {text!r}")


def summarize_sales(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Sales file not found: {path}")
    seen = set()
    by_category, by_month, by_product = defaultdict(float), defaultdict(float), defaultdict(float)
    total, skipped = 0.0, 0
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            order_id = row["order_id"].strip()
            if order_id in seen:
                continue
            seen.add(order_id)
            if row["status"].strip().lower() != "completed":
                continue
            qty, price = _positive_number(row["quantity"]), _positive_number(row["unit_price"])
            if qty is None or price is None:
                skipped += 1
                continue
            revenue = qty * price
            total += revenue
            by_category[row["category"].strip().lower()] += revenue
            by_month[_parse_date(row["order_date"]).strftime("%Y-%m")] += revenue
            by_product[row["product"].strip()] += revenue
    ranked = sorted(by_product.items(), key=lambda kv: (-round(kv[1], 2), kv[0]))
    return {
        "total_revenue": round(total, 2),
        "revenue_by_category": {k: round(v, 2) for k, v in by_category.items()},
        "revenue_by_month": {k: round(v, 2) for k, v in by_month.items()},
        "top_product": ranked[0][0] if ranked else None,
        "rows_skipped": skipped,
    }

Writing hidden_tests/reference_solution.py


In [8]:
import importlib.util

spec = importlib.util.spec_from_file_location("reference_solution", TESTDIR / "reference_solution.py")
reference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reference)

EXPECTED = {name: {"path": str((TESTDIR / f"{name}.csv").resolve()),
                   "result": reference.summarize_sales(TESTDIR / f"{name}.csv")}
            for name in ("sales", "check", "empty")}
(TESTDIR / "expected.json").write_text(json.dumps(EXPECTED, indent=2))

TEST_SCRIPT = r'''
import json, sys
sys.path.insert(0, ".")
EXPECTED = json.load(open("expected.json"))
results = []

def record(name, passed, detail=""):
    results.append({"test": name, "passed": passed, "detail": detail})

def compare(got, exp):
    if not isinstance(got, dict):
        return [f"returned {type(got).__name__}, expected dict"]
    problems = []
    if set(got) != set(exp):
        problems.append(f"keys are {sorted(got)}, expected {sorted(exp)}")
    for key, want in exp.items():
        have = got.get(key)
        if isinstance(want, dict):
            try:
                ok = isinstance(have, dict) and set(have) == set(want) and all(
                    abs(float(have[k]) - want[k]) < 0.005 for k in want)
            except (TypeError, ValueError):
                ok = False
        elif isinstance(want, float):
            try:
                ok = abs(float(have) - want) < 0.005
            except (TypeError, ValueError):
                ok = False
        else:
            ok = have == want
        if not ok:
            problems.append(f"{key}: expected {want!r}, got {have!r}")
    return problems

try:
    from generated_solution import summarize_sales
    record("1. code imports without errors", True)
except Exception as e:
    record("1. code imports without errors", False, f"{type(e).__name__}: {e}")
    print(json.dumps(results))
    sys.exit()

for name, key in [("2. correct summary of sales.csv", "sales"),
                  ("3. correct summary of a different file (no hard-coding)", "check"),
                  ("4. header-only file returns an empty summary", "empty")]:
    try:
        problems = compare(summarize_sales(EXPECTED[key]["path"]), EXPECTED[key]["result"])
        record(name, not problems, "; ".join(problems))
    except Exception as e:
        record(name, False, f"raised {type(e).__name__}: {e}")

name = "5. missing file raises FileNotFoundError naming the path"
try:
    summarize_sales("no_such_file.csv")
    record(name, False, "no exception was raised")
except FileNotFoundError as e:
    named = "no_such_file.csv" in str(e)
    record(name, named, "" if named else "message does not include the path")
except Exception as e:
    record(name, False, f"raised {type(e).__name__} instead of FileNotFoundError")

print(json.dumps(results))
'''
(TESTDIR / "run_tests.py").write_text(TEST_SCRIPT)


def run_tests(solution_code):
    # Run the test suite against a candidate solution in a separate process
    (TESTDIR / "generated_solution.py").write_text(solution_code)
    try:
        proc = subprocess.run([sys.executable, "-B", "run_tests.py"], cwd=TESTDIR,
                              capture_output=True, text=True, timeout=60)
        return json.loads(proc.stdout.strip().splitlines()[-1])
    except Exception as e:
        return [{"test": "test run", "passed": False,
                 "detail": f"the tests could not run ({type(e).__name__}); the code may crash or hang on import"}]


def show_tests(results):
    for r in results:
        print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['test']}" + (f"\n        {r['detail']}" if r["detail"] else ""))
    print(f"  => {sum(r['passed'] for r in results)}/{len(results)} tests passed")


print("Expected answer for sales.csv (from my reference solution):")
print(json.dumps(EXPECTED["sales"]["result"], indent=2))
print("\nSanity check, the reference solution passes its own tests:")
show_tests(run_tests((TESTDIR / "reference_solution.py").read_text()))

Expected answer for sales.csv (from my reference solution):
{
  "total_revenue": 3492.71,
  "revenue_by_category": {
    "accessories": 305.94,
    "electronics": 1916.8,
    "home office": 1269.97
  },
  "revenue_by_month": {
    "2026-07": 1407.48,
    "2026-08": 1598.78,
    "2026-09": 486.45
  },
  "top_product": "UltraBook 14",
  "rows_skipped": 4
}

Sanity check, the reference solution passes its own tests:
  PASS  1. code imports without errors
  PASS  2. correct summary of sales.csv
  PASS  3. correct summary of a different file (no hard-coding)
  PASS  4. header-only file returns an empty summary
  PASS  5. missing file raises FileNotFoundError naming the path
  => 5/5 tests passed


## The ReAct prompt

Two parts:
- **`REACT_SYSTEM`** defines the ReAct cycle: the stages (PLAN, THOUGHT, ACTION, OBSERVATION, FIX, FINAL_ANSWER), the exact reply format, and how the one tool (`run_python`) behaves.
- **`TASK_SPEC`** is the coding task: inputs, outputs, rules, error handling, allowed libraries and formatting.

In [9]:
REACT_SYSTEM = """You are a careful Python developer who solves coding tasks with the ReAct method: you alternate between reasoning and acting, and you never assume what a file contains without checking it.

You have one tool, run_python. When you send an ACTION, its code runs in a fresh Python process in a folder that contains the task's input file. The next message you receive is the OBSERVATION: everything the code printed to stdout and stderr, including tracebacks. Nothing carries over between actions, so every ACTION must include its own imports. Use print() to see results.

Work in these stages:
1. PLAN (first reply only): a numbered plan, max 5 steps. Then give your first THOUGHT and ACTION.
2. THOUGHT: before every ACTION, 1 to 3 sentences on what you learned from the last OBSERVATION and what this ACTION will check.
3. ACTION: exactly one ```python code block. Keep actions small: inspect the data, test one idea, or test a draft function on the real file.
4. OBSERVATION: sent back to you by the system. Read it carefully before continuing.
5. FIX: if an OBSERVATION shows an error or a wrong result, start your THOUGHT with "FIX:" and name the cause before you change the code.
6. FINAL_ANSWER: only after you have run your draft function on the real file and checked its output. Give the complete code in one ```python code block. It is tested automatically; failed tests come back as an OBSERVATION and you continue with FIX.

Reply format (use these exact labels):
PLAN: ...            (first reply only)
THOUGHT: ...
ACTION:
```python
...
```
or, when you are done:
THOUGHT: ...
FINAL_ANSWER:
```python
...
```
Send only one ACTION or FINAL_ANSWER per reply. Never write an OBSERVATION yourself; wait for the real one."""


TASK_SPEC = """TASK
Write a Python function `summarize_sales(path: str) -> dict` that summarizes an online store's order export.

INPUT
- `path`: path to a CSV file with the columns order_id, order_date, product, category, quantity, unit_price, status.
- The file for this task is `sales.csv` in your working folder. It comes from a real export and may contain formatting problems. You have not seen it yet, so inspect it with ACTION steps before writing the function.

OUTPUT: a dict with exactly these keys
- "total_revenue": float, the sum of quantity * unit_price over all counted rows
- "revenue_by_category": dict of {category: revenue}; category names lowercased, with surrounding spaces removed
- "revenue_by_month": dict of {"YYYY-MM": revenue}
- "top_product": the product name (surrounding spaces removed) with the highest revenue; ties go to the alphabetically first name; None if no rows are counted
- "rows_skipped": int (see rule 3)
Round every money value to 2 decimals.

RULES
1. Duplicate rows have the same order_id. Count each order_id only once (keep the first one).
2. Only count rows whose status is "completed" (ignore case and surrounding spaces).
3. Of the remaining rows, skip any row whose quantity or unit_price cannot be converted to a number greater than 0, and count it in "rows_skipped". Duplicate and non-completed rows are NOT counted in rows_skipped.
4. Dates may appear in more than one format. Dates with slashes are US style (month/day/year). Every date in the file is valid.
5. Never hard-code values from sales.csv. The function will be tested on other files with the same kinds of problems.

ERROR HANDLING
- If `path` does not exist, raise FileNotFoundError with a message that includes the path.
- A file with only the header row returns total_revenue 0.0, empty dicts, top_product None, and rows_skipped 0.

CONSTRAINTS
- Python 3 standard library and pandas only. No network access, no installing packages, and do not create, modify or delete files.
- The FINAL_ANSWER code must be self-contained: all imports at the top, type hints, a docstring for summarize_sales, helper functions allowed, and no print statements or other code outside function definitions."""

print(f"REACT_SYSTEM: {len(REACT_SYSTEM.split())} words | TASK_SPEC: {len(TASK_SPEC.split())} words")

REACT_SYSTEM: 293 words | TASK_SPEC: 347 words


## Iteration 1 (baseline): one-shot prompt, no ReAct

Before building the loop, I tested the plain approach: the **same task description**, but the model has to write the final function in a single reply, without inspecting the file or running any code. This shows what the ReAct cycle adds.

In [ ]:
BASELINE_SYSTEM = "You are a careful Python developer."
BASELINE_NOTE = ("\n\nNOTE FOR THIS ATTEMPT: you cannot inspect the file or run any code. "
                 "Write the complete final code now in one ```python code block.")


def extract_code(text):
    blocks = re.findall(r"```[A-Za-z0-9_+-]*[ \t]*\n(.*?)```", text, re.DOTALL)  # ```python, ```py, or plain ```
    return blocks[0] if blocks else None


baseline_reply = generate(TASK_SPEC + BASELINE_NOTE, system=BASELINE_SYSTEM)
baseline_code = extract_code(baseline_reply) or ""
show("BASELINE | code written in one shot (no inspection, no execution)", baseline_code or baseline_reply)

baseline_results = run_tests(baseline_code)
print("\nBASELINE | automatic test results")
show_tests(baseline_results)

## Iteration 2: the ReAct loop

`run_python` is the model's only tool. It saves the ACTION code to a file and runs it in a separate Python process inside `react_workspace/` (which contains only `sales.csv`), with a 30-second timeout. A simple safety check blocks code that tries to start processes, use the network, delete files, or leave that folder (so it cannot peek at the hidden tests).

`react_loop` sends the model's replies and the real observations back and forth until the final function passes every test, or a step limit is reached.

In [ ]:
BLOCKED = ["subprocess", "os.system", "os.popen", "shutil", "os.remove", "os.unlink", "rmdir",
           "requests", "urllib", "socket", "http.client", "pip install", "os.chdir",
           "hidden_tests", "expected.json", "reference_solution", "../", "'..'", '".."']


def run_python(code_text, timeout=30):
    """The model's tool: run ACTION code in a separate process and return what it printed."""
    banned = [b for b in BLOCKED if b in code_text]
    if banned:
        return f"BLOCKED: this action uses {banned}, which is not allowed. Only read the CSV and print results."
    script = WORKDIR / "_action.py"
    script.write_text(code_text)
    try:
        proc = subprocess.run([sys.executable, "-B", script.name], cwd=WORKDIR,
                              capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return f"ERROR: the code did not finish within {timeout} seconds."
    finally:
        script.unlink(missing_ok=True)
    output = (proc.stdout + ("\n" + proc.stderr if proc.stderr else "")).strip()
    if not output:
        return "(the code ran without errors but printed nothing; use print() to see results)"
    return output if len(output) <= 3000 else output[:1000] + "\n...[output truncated]...\n" + output[-2000:]


def parse_reply(reply):
    """Split a model reply into its reasoning text, its kind (action / final / invalid) and its code."""
    reply = re.split(r"\n\s*OBSERVATION\s*:", reply)[0]   # ignore any observation the model invents
    final = re.search(r"FINAL_ANSWER\s*:?", reply)
    action = re.search(r"ACTION\s*:?", reply)
    marker = final or action
    reasoning = reply[:marker.start()].strip() if marker else reply.strip()
    code_text = extract_code(reply[marker.start():]) if marker else None
    if final and code_text:
        kind = "final"
    elif action and code_text:
        kind = "action"
    else:
        kind = "invalid"
    return reply, reasoning, kind, code_text


def stage_label(step, reasoning):
    if step == 1:
        return "PLAN + THOUGHT"
    return "FIX" if re.search(r"THOUGHT\s*:\s*FIX", reasoning, re.IGNORECASE) or reasoning.upper().startswith("FIX") else "THOUGHT"


def react_loop(max_steps=12):
    history = [user_turn(TASK_SPEC)]
    transcript = []
    for step in range(1, max_steps + 1):
        reply, reasoning, kind, code_text = parse_reply(generate(history, system=REACT_SYSTEM))
        history.append(model_turn(reply))
        label = stage_label(step, reasoning)
        show(f"STEP {step} | {label}", reasoning)

        if kind == "action":
            show(f"STEP {step} | ACTION (code the model wrote)", code_text)
            observation = run_python(code_text)
            show(f"STEP {step} | OBSERVATION (what happened when Colab ran it)", observation)
        elif kind == "final":
            show(f"STEP {step} | FINAL_ANSWER (complete function)", code_text)
            results = run_tests(code_text)
            print(f"\nSTEP {step} | AUTOMATIC TESTS")
            show_tests(results)
            transcript.append({"step": step, "stage": label, "kind": kind, "reasoning": reasoning,
                               "code": code_text, "tests_passed": sum(r["passed"] for r in results)})
            if all(r["passed"] for r in results):
                return {"passed": True, "steps": step, "final_code": code_text,
                        "test_results": results, "transcript": transcript}
            failures = "\n".join(f"- {r['test']}: {r['detail']}" for r in results if not r["passed"])
            observation = "Your FINAL_ANSWER failed these automatic tests:\n" + failures
            show(f"STEP {step} | OBSERVATION (test failures sent back to the model)", observation)
        else:
            observation = ("FORMAT ERROR: each reply needs THOUGHT followed by either ACTION or FINAL_ANSWER, "
                           "each with exactly one ```python code block.")
            show(f"STEP {step} | OBSERVATION (format problem)", observation)

        if kind != "final":
            transcript.append({"step": step, "stage": label, "kind": kind, "reasoning": reasoning,
                               "code": code_text, "observation": observation})
        history.append(user_turn("OBSERVATION:\n" + observation))
    return {"passed": False, "steps": max_steps, "final_code": None, "test_results": None, "transcript": transcript}

### Run the ReAct loop
Every block below is labeled with its stage, so the full reason → code → run → observe → fix cycle is visible.

In [ ]:
react = react_loop()
print("\n" + "=" * 100)
print(f"RESULT: {'all tests passed' if react['passed'] else 'did not pass all tests'} after {react['steps']} steps")

## Final result: the generated function in use
The final code is saved to `summarize_sales.py`, then imported and run like any normal Python module.

In [ ]:
if not react["passed"]:
    raise RuntimeError("The ReAct loop did not finish. Re-run the loop cell (it usually succeeds on a retry).")

Path("summarize_sales.py").write_text(react["final_code"])
import importlib, summarize_sales as generated
importlib.reload(generated)

summary = generated.summarize_sales("react_workspace/sales.csv")
show("summarize_sales('sales.csv') output", summary)

print("\nError handling check:")
try:
    generated.summarize_sales("missing.csv")
except FileNotFoundError as e:
    print(f"  FileNotFoundError: {e}")

In [ ]:
import pandas as pd

fix_steps = [t for t in react["transcript"] if t["stage"] == "FIX"]
comparison = pd.DataFrame([
    {"approach": "Iteration 1: one-shot prompt (no ReAct)",
     "tests_passed": f"{sum(r['passed'] for r in baseline_results)}/{len(baseline_results)}",
     "model_calls": 1, "code_runs_observed": 0, "fix_steps": 0},
    {"approach": "Iteration 2: ReAct loop",
     "tests_passed": f"{sum(r['passed'] for r in react['test_results'])}/{len(react['test_results'])}",
     "model_calls": react["steps"],
     "code_runs_observed": sum(t["kind"] == "action" for t in react["transcript"]),
     "fix_steps": len(fix_steps)},
])
display(comparison)
print("Model(s) used:", ", ".join(sorted(MODELS_USED)))

### Fix log (generated from the transcript)
Every step where the model observed a problem and fixed it:

In [ ]:
if not fix_steps:
    print("The model did not need a FIX step on this run: its first draft passed when it tested it.")
for t in fix_steps:
    print(f"STEP {t['step']}:")
    print(textwrap.indent(wrap(t["reasoning"], 95), "  "))
    print()

## Prompts used (for the write-up)

In [ ]:
for name, prompt in [("ReAct system prompt (REACT_SYSTEM)", REACT_SYSTEM),
                     ("Task prompt (TASK_SPEC)", TASK_SPEC),
                     ("Baseline system prompt", BASELINE_SYSTEM),
                     ("Baseline note added to TASK_SPEC", BASELINE_NOTE.strip())]:
    print(f"==================== {name} ====================\n{prompt}\n")

## Notes

**How the ReAct stages map to the notebook**
- **Reason:** the model writes a PLAN, then a THOUGHT before every action.
- **Act:** each ACTION is real Python code that `run_python` executes in Colab.
- **Observe:** the printed output or traceback goes back to the model word for word as an OBSERVATION.
- **Fix:** when an observation shows a problem, the model labels its THOUGHT "FIX:" and names the cause before changing the code. Failed automatic tests on a FINAL_ANSWER also come back as observations.

**Testing and iteration**
- Iteration 1 used the same requirements in a one-shot prompt; the comparison table shows how many tests it passed without being able to look at the data.
- Iteration 2 (ReAct) had to inspect the file, run drafts and fix what broke; the fix log above lists each fix.
- Correctness is judged by automatic tests against my own reference answers, including a second file to catch hard-coding, an empty file, and a missing file.

**Limitations:** the step limit is 12, the safety check is a simple blocklist rather than a real sandbox, and results vary between runs because the model's output is not deterministic.